# AA 279B HW 6 Prob 2

## Imports

In [119]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [120]:
import matlab.engine
from matlab.engine import MatlabEngine

# 1. Start the MATLAB process in the background
print("Starting MATLAB engine...")
eng: MatlabEngine = matlab.engine.start_matlab()

Starting MATLAB engine...


## Setup

In [121]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

In [122]:
import numpy as np
from numpy.linalg import norm

from constants import DAY_TO_SEC, RAD_TO_SIDEREAL_HOUR, RAD_TO_DEG
from orbital import calc_theta_g, rv_HCI, ECI_from_HCI
from spacetime import greg_to_jd
from angle import wrap_to_2pi, wrap_to_pi, angle_between

In [123]:
MU_SUN = 1.3271244004193938e11  # km3/s2
ECLIPTIC_ANGLE = 23.45  # DEG
W_EARTH = 0.0000729211585530  # rad/s

In [124]:
# Dates
t_earth_dep_jd = greg_to_jd(2026, 10, 19, 19, 41, 3)
t_mars_arr_jd = greg_to_jd(2027, 8, 15, 0, 0, 0)
t_trans_jd = (t_mars_arr_jd - t_earth_dep_jd) * DAY_TO_SEC

# Greenwich angle
t_ref_jd = greg_to_jd(2026, 1, 1, 0, 0, 0) 
theta_ref = 6.7107222 / RAD_TO_SIDEREAL_HOUR

# Launch site
launch_lat_deg = 30.2333
launch_lon_deg = 130.9776


### Part a

In [125]:
r_earth_dep_hci, v_earth_dep_hci = rv_HCI(3, t_earth_dep_jd, eng)
r_mars_arr_hci, v_mars_arr_hci = rv_HCI(4, t_mars_arr_jd, eng)

def calc_transfer(r1_hci: np.ndarray, r2_hci: np.ndarray, t_trans: float):
    result = eng.AA279lambert_curtis(MU_SUN, r1_hci, r2_hci, "pro", "ignored", t_trans, nargout=2)
    
    # heliocentric asymptotic arrival and departure velocity vectors
    v1_hci, v2_hci = np.array(result[0]).reshape((3,)), np.array(result[1]).reshape((3,))

    return v1_hci, v2_hci

v_dep_hci, v_arr_hci = calc_transfer(r_earth_dep_hci, r_mars_arr_hci, t_trans_jd)

print(f"Heliocentric Earth Departure Velocity:\n\t{v_dep_hci} ({norm(v_dep_hci):.3f})[km/s]")
print(f"Heliocentric Mars Arrival Velocity:\n\t{v_arr_hci} ({norm(v_arr_hci):.3f})[km/s]")

Heliocentric Earth Departure Velocity:
	[-15.79181262  28.89963785   0.07344544] (32.933)[km/s]
Heliocentric Mars Arrival Velocity:
	[ 17.17778138 -12.41740268  -0.04199491] (21.196)[km/s]


### Part b

In [126]:
# Earth-centric
v_dep_eci = ECI_from_HCI() @ (v_dep_hci - v_earth_dep_hci)
print(f"Earth-centric Departure Velocity:\n\t{v_dep_eci} ({norm(v_dep_eci):.3f})[km/s]")

Earth-centric Departure Velocity:
	[-2.21913846 -0.05795613 -2.2526643 ] (3.163)[km/s]


### Part c

In [127]:
# C3, RLA, DLA
C3 = norm(v_dep_eci)**2
print(f"Earth C3:\n\t{C3:.3f} [km2/s2]")

# decl = np.arctan2(v_dep_eci[2], norm(v_dep_eci[0:2]))  # also works
dla = wrap_to_pi(np.arcsin(v_dep_eci[2] / norm(v_dep_eci)))
print(f"Declination:\n\t{dla * RAD_TO_DEG:.3f} [deg]")

rla = wrap_to_2pi(np.arctan2(v_dep_eci[1], v_dep_eci[0]))
print(f"Ascension:\n\t{rla * RAD_TO_DEG:.3f} [deg]")

Earth C3:
	10.002 [km2/s2]
Declination:
	-45.420 [deg]
Ascension:
	181.496 [deg]


### Part d

In [128]:
alpha_launch = wrap_to_2pi((launch_lon_deg / RAD_TO_DEG) + calc_theta_g(t_earth_dep_jd * DAY_TO_SEC, theta_ref, t_ref_jd * DAY_TO_SEC, W_EARTH))
print(f"Launch site ascension: {alpha_launch * RAD_TO_DEG:.3f}º")
# tan of azimuth
tan_beta = np.sin(rla - launch_lon_deg/RAD_TO_DEG) / (
    np.cos(launch_lat_deg/RAD_TO_DEG)*np.tan(dla) - np.sin(launch_lat_deg/RAD_TO_DEG)*np.cos(rla - launch_lon_deg/RAD_TO_DEG)
)

betas = (
    wrap_to_2pi(np.arctan(tan_beta)),
    wrap_to_2pi(np.arctan(tan_beta) + np.pi)
)
print(f"Solution 1: {betas[0] * RAD_TO_DEG:.3f}º")
print(f"Solution 2: {betas[1] * RAD_TO_DEG:.3f}º")

Launch site ascension: 94.533º
Solution 1: 327.184º
Solution 2: 147.184º


In [132]:
from plotting import plot_earth_departure

fig = plot_earth_departure(
    v_dep_eci,
    launch_lat_deg,
    launch_lon_deg,
    alpha_launch=alpha_launch,
)
